## Inspect fossil shape completion and identification results
---
*Last edited 11 March 2026 by K. Wolcott*

### Shape completion

In [ ]:
# Load in meshes

# Imports
import sys
import os
import pyvista as pv
pv.set_jupyter_backend("trame")
import random

# Case insensitive file matching
def find_ci_file(directory, name):
    if not os.path.isdir(directory):
        return None
    nl = name.lower()
    for f in os.listdir(directory):
        if f.lower() == nl:
            return os.path.join(directory, f)
    return None

# Define training directory used for shape completion
repo_wd = "/path/to/your/repo/dir/NSM/nsm/" # TO DO: # adjust to wherever the NSM repo root is
TRAIN_DIR = "run_v57" # TO DO: Choose training directory containing model ckpt and latent codes
if not os.getcwd().endswith(TRAIN_DIR):
    os.chdir(repo_wd + TRAIN_DIR)
print(f"Working directory set to {os.getcwd()}")

# Randomly inspect or by filename
random_mesh = True # TO DO: Randomly inspect a fossil? (True or False)
if random_mesh == False:
    fossil_name = "ZZZZ_DMNH-EPV-142201-MAD0720-1_hollow_align" # TO DO: Define fossil to inspect
else:
    mesh_dir = "fossils/models_smooth_hollow/aligned/"
    mesh_list = os.listdir(mesh_dir)
    fossil_name = random.sample(mesh_list, 1)[0]
    fossil_name = os.path.splitext(fossil_name)[0]
print("\nBuilding mesh paths...\n")

# Original smoothed fossil mesh
folder = "models_smooth"
mesh1_path = "fossils/" + folder + "/aligned/" 
mesh1_fname = fossil_name.replace("_hollow", "") + ".vtk"
print("Path to original fossil: ", mesh1_path+mesh1_fname)

# Smooth hollow fossil mesh
folder = "models_smooth_hollow" if "hollow" in fossil_name else "models_smooth"
mesh2_path = "fossils/" + folder + "/aligned/" 
mesh2_fname = fossil_name + ".vtk"
print("Path to smooth_hollow mesh: ", mesh2_path+mesh2_fname)

# Shape completion mesh
mesh3_path = "shape_completion/predictions/" + fossil_name + "/"
mesh3_fname = fossil_name + "_shape_completion.vtk"
print("Path to shape completed mesh: ", mesh3_path+mesh3_fname)

# Case insensitive file matching
print("\nPerforming case-insensitive filepath matching...\n")
mesh1_full = find_ci_file(mesh1_path, mesh1_fname)
mesh2_full = find_ci_file(mesh2_path, mesh2_fname)
mesh3_full = find_ci_file(mesh3_path, mesh3_fname)

# Load in meshes
print("Loading original mesh from: ", mesh1_full)
print("Loading preprocessed mesh from: ", mesh2_full)
print("Loading shape completed mesh from: ", mesh3_full)
mesh1 = pv.read(mesh1_full)
mesh1 = mesh1.triangulate().decimate(0.9)
mesh2 = pv.read(mesh2_full)
mesh2 = mesh2.triangulate().decimate(0.9)
mesh3 = pv.read(mesh3_full)
mesh3 = mesh3.triangulate().decimate(0.9)
meshes = [mesh1, mesh2, mesh3]

In [ ]:
# Plot the meshes interactively
titles = [mesh1_fname, mesh2_fname, mesh3_fname]
plotter = pv.Plotter(shape=(1, 3), border=False)
for i, mesh in enumerate(meshes):
    plotter.subplot(0, i)
    if mesh is not None:
        plotter.add_mesh(mesh, color="lightgray", opacity=1.0)
        plotter.add_axes()
        plotter.add_text(titles[i], font_size=10)
    else:
        plotter.add_text("Missing mesh")

# Sync camera across views
plotter.link_views()
plotter.show()

### Classification

In [ ]:
# Parse classificaiton results
import re
import json

# Load model config file
def load_config(config_path='model_params_config.json'):
    try:
        with open(config_path, 'r') as f:
            config = json.load(f)
        print(f"\033[92mLoaded config from {config_path}\033[0m")
        return config
    except FileNotFoundError:
        raise FileNotFoundError(f"Error: model_params_config.json not found at {config_path}")

# Get top 5 matches from similar_meshes_pca_regularized_95pct_cos.txt
def parse_top5_matches(txt_path):
    matches = []
    with open(txt_path, "r") as f:
        for line in f:
            m = re.match(r"Name:\s*(.+?),\s*Index:\s*(\d+),\s*Distance:\s*([\d.]+)", line.strip())
            if m:
                matches.append({"name":     m.group(1),
                                "index":    int(m.group(2)),
                                "distance": float(m.group(3)),})
    return matches

# Load config
config = load_config(config_path='model_params_config.json')
device = config.get("device", "cuda:0")
train_paths = config['list_mesh_paths']

# Load in classification results
shape_completed_mesh = True # TO DO: Inspect for shape completed mesh or original?
if shape_completed_mesh == True:
    fossil_path = fossil_name + "_shape_completion"
    results_dir = "classify_vertebrae/predictions/" + fossil_path + "/"
else:
    fossil_path = fossil_name
    results_dir = "classify_vertebrae/predictions/" + fossil_name + "/"

# Load in top 5 closest meshes file
top5_matches_path = results_dir + "similar_meshes_pca_regularized_95pct_cos.txt"

# Get filepaths for top 5 closest meshes from train_paths
matches = parse_top5_matches(top5_matches_path)

# Resolve full paths from train_paths using index
for match in matches:
    idx = match["index"]
    match["filepath"] = train_paths[idx] if idx < len(train_paths) else None
    # Parse family and genus from filename e.g. "varanidae_varanus_komodoensis_..."
    parts = match["name"].replace(".vtk", "").split("_")
    match["family"] = parts[0] if len(parts) > 0 else None
    match["genus"]  = parts[1] if len(parts) > 1 else None
# Extract filepaths for top 5 matches
match_paths = [m["filepath"] for m in matches]

# Print summary
print("Top 5 closest matches for: ", fossil_name)
for m in matches:
    print(f"  {m['family']:20s} {m['genus']:20s}  dist={m['distance']:.4f}  {m['filepath']}")

In [ ]:
# Inspect latent space plots for top 5 closest matches
from IPython.display import display, Image

# Load in latent space plots for top 5 closest meshes
pca_path = results_dir + "latent_space_pca_pca_regularized_95pct_cos.png"
tsne_path = results_dir + "latent_space_tsne_pca_regularized_95pct_cos.png"

print("\nPCA latent space:")
display(Image(filename=pca_path, width=800))

print("\nt-SNE latent space:")
display(Image(filename=tsne_path, width=800))

In [ ]:
# Inspect decoded mesh vs input mesh

# Load in encoded and decoded representation of input mesh (Good for inspecting if something weird is going on with model encoding/decoding)
decoded_mesh_fname = results_dir + fossil_name + "_shape_completion_decoded_novel_pca_regularized_95pct_cos.vtk"
decoded_mesh = pv.read(decoded_mesh_fname)
decoded_mesh = decoded_mesh.decimate(0.9)

# Compare with original shape completed mesh
completed_mesh = mesh3

# Plot the meshes interactively
meshes = [completed_mesh, decoded_mesh]
titles = [mesh3_fname, decoded_mesh_fname]
plotter = pv.Plotter(shape=(1, 2), border=False)
for i, mesh in enumerate(meshes):
    plotter.subplot(0, i)
    if mesh is not None:
        plotter.add_mesh(mesh, color="lightgray", opacity=1.0)
        plotter.add_axes()
        plotter.add_text(titles[i], font_size=10)
    else:
        plotter.add_text("Missing mesh")

# Sync camera across views
plotter.link_views()
plotter.show()

In [ ]:
# Inspect closest matching meshes against fossil

# Extract filepaths for top 5 matches
match_paths = [m["filepath"] for m in matches]
titles = [f"Top{i+1}: {m['filepath']}" for i, m in enumerate(matches)]

# Inpsect idx
inspect_idx = 1 # TO DO: Choose to inspect top 1, 2, 3, 4, 5 best match

# Select matching mesh
match_path = match_paths[inspect_idx - 1]
match_title = titles[inspect_idx - 1]
match_mesh = pv.read(match_path)

# Build list to plot
meshes = [mesh1, mesh3, match_mesh]

# Plot the meshes interactively
titles = [mesh1_fname, mesh3_fname, match_title]
plotter = pv.Plotter(shape=(1, 3), border=False)
for i, mesh in enumerate(meshes):
    plotter.subplot(0, i)
    if mesh is not None:
        plotter.add_mesh(mesh, color="lightgray", opacity=1.0)
        plotter.add_axes()
        plotter.add_text(titles[i], font_size=10)
    else:
        plotter.add_text("Missing mesh")

# Sync camera across views
plotter.link_views()
plotter.show()